# GPS для OD-потоков (v7)

**Изменения v7 vs v6:**
1. **Унификация multi-city**: val CPC по полной матрице, выбор лучших/последних весов, единый формат
2. **LGBM pipeline**: GPS-эмбеддинги → LightGBM, перебор донорских моделей
3. **NaN protection**: пропуск батчей с NaN, прерывание эксперимента при дивергенции
4. **Сохранение**: метрики → CSV, веса → .pt файлы после каждого эксперимента
5. **Все улучшения v6**: SPE/RRWP, GraphNorm/GRANOLA, ZINB, Log-transform, гранулярные эксперименты

In [ ]:
!pip install torch_geometric scikit-learn scipy geopandas lightgbm -q

## Импорты и конфигурация

In [ ]:
import os, time, warnings, csv
from dataclasses import dataclass, field, replace, asdict
from typing import Optional
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BatchNorm1d, Linear, ModuleList, ReLU, Sequential

from sklearn.preprocessing import MinMaxScaler
from scipy.stats import gaussian_kde
import lightgbm as lgb

from torch_geometric.nn import GINEConv, GPSConv
from torch_geometric.data import Data
import torch_geometric.transforms as T
import geopandas as gpd

# ─── Пути ────────────────────────────────────────────────────────────────────
DATA_PATH      = "/content/CommutingODGen-Dataset/data"
SHP_PATH       = "/content/CommutingODGen-Dataset/assets/Boundaries_Regions_within_Areas"
RESULTS_DIR    = Path("results")
WEIGHTS_DIR    = RESULTS_DIR / "weights"
METRICS_CSV    = RESULTS_DIR / "metrics.csv"
RESULTS_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR.mkdir(exist_ok=True)

SINGLE_CITY_ID = "48201"
MULTI_CITY_IDS = ["17031","48201","04013","06073","06059","36047","12086","48113","06065","36081"]

# ─── Архитектура ─────────────────────────────────────────────────────────────
HIDDEN_DIM=64; PE_DIM=8; PE_WALK_LEN=20; GPS_HEADS=4; GPS_LAYERS=4; GPS_DROPOUT=0.1
TF_HEADS=4; TF_LAYERS=2; TF_DROPOUT=0.1

# ─── Обучение ────────────────────────────────────────────────────────────────
EPOCHS=50; LEARNING_RATE=1e-3; PATIENCE=30; ORIGIN_BATCH_SIZE=32; DEST_BATCH_SIZE=256
N_DEST_SAMPLE=128; MC_EPOCHS=30

# ─── Loss ────────────────────────────────────────────────────────────────────
HUBER_DELTA=1.0; HUBER_KDE_BW=2.0; HUBER_MIN_PROB=1e-4
LAMBDA_MAIN=0.5; LAMBDA_SUB=0.5; NORMALIZE_MULTITASK=True

# ─── Фичи ────────────────────────────────────────────────────────────────────
USE_LU_FEATURES=False; USE_JOBS_FEATURES=False

# ─── NaN protection ──────────────────────────────────────────────────────────
NAN_BATCH_THRESHOLD = 0.5  # доля NaN-батчей для прерывания эксперимента

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


@dataclass
class TrainingConfig:
    decoder_type:       str   = 'transflower'
    loss_type:          str   = 'huber'
    prediction_mode:    str   = 'raw'
    pe_type:            str   = 'rwpe'
    gps_norm_type:      str   = 'batch_norm'
    use_log_transform:  bool  = False
    use_dest_sampling:  bool  = True
    n_dest_sample:      int   = N_DEST_SAMPLE
    include_zero_pairs: bool  = True
    zero_pair_ratio:    float = 0.3
    epochs:             int   = EPOCHS
    learning_rate:      float = LEARNING_RATE
    patience:           int   = PATIENCE
    mc_epochs:          int   = MC_EPOCHS
    mc_val_cpc_sample:  int   = 512

    def describe(self):
        parts = [f"GPS+{self.decoder_type}+{self.loss_type}+{self.prediction_mode}",
                 f"pe={self.pe_type}", f"norm={self.gps_norm_type}"]
        if self.use_log_transform: parts.append("log")
        parts.append(f"zeros={self.include_zero_pairs} samp={self.use_dest_sampling}")
        return " | ".join(parts)


# ─── Сохранение результатов ──────────────────────────────────────────────────
def save_metrics_to_csv(run_id, run_name, config, metrics_full, metrics_nz,
                        metrics_test, n_params, epochs_trained, status='ok'):
    """Дописывает строку метрик в CSV."""
    row = {
        'timestamp': datetime.now().isoformat(),
        'run_id': run_id, 'name': run_name, 'status': status,
        'decoder': config.decoder_type, 'loss_type': config.loss_type,
        'prediction_mode': config.prediction_mode,
        'pe_type': config.pe_type, 'gps_norm_type': config.gps_norm_type,
        'use_log_transform': config.use_log_transform,
        'n_params': n_params, 'epochs_trained': epochs_trained,
        'CPC_full': metrics_full['CPC'], 'CPC_nz': metrics_nz['CPC'],
        'CPC_test': metrics_test['CPC'],
        'MAE_full': metrics_full['MAE'], 'RMSE_full': metrics_full['RMSE'],
    }
    file_exists = METRICS_CSV.exists()
    with open(METRICS_CSV, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists: writer.writeheader()
        writer.writerow(row)
    print(f"  → Метрики сохранены в {METRICS_CSV}")


def save_model_weights(run_id, model):
    """Сохраняет веса модели."""
    path = WEIGHTS_DIR / f"{run_id}.pt"
    torch.save(model.state_dict(), path)
    print(f"  → Веса сохранены в {path}")


print("TrainingConfig ready.")
print(" ", TrainingConfig().describe())


## Определение фичей

In [ ]:
ALL_DEMO_FEATURE_NAMES = [
    "pop_total","pop_male","pop_female",
    "age_under5_total","age_under5_male","age_under5_female",
    "age_5to9_total","age_5to9_male","age_5to9_female",
    "age_10to14_total","age_10to14_male","age_10to14_female",
    "age_15to19_total","age_15to19_male","age_15to19_female",
    "age_20to24_total","age_20to24_male","age_20to24_female",
    "age_25to29_total","age_25to29_male","age_25to29_female",
    "age_30to34_total","age_30to34_male","age_30to34_female",
    "age_35to39_total","age_35to39_male","age_35to39_female",
    "age_40to44_total","age_40to44_male","age_40to44_female",
    "age_45to49_total","age_45to49_male","age_45to49_female",
    "age_50to54_total","age_50to54_male","age_50to54_female",
    "age_55to59_total","age_55to59_male","age_55to59_female",
    "age_60to64_total","age_60to64_male","age_60to64_female",
    "age_65to69_total","age_65to69_male","age_65to69_female",
    "age_70to74_total","age_70to74_male","age_70to74_female",
    "age_75to79_total","age_75to79_male","age_75to79_female",
    "age_80to84_total","age_80to84_male","age_80to84_female",
    "age_85plus_total","age_85plus_male","age_85plus_female",
    "median_age_total","median_age_male","median_age_female","median_earnings",
    "worker_private_wage","worker_government","worker_self_employed","worker_unpaid_family",
    "mean_travel_time_min",
    "vehicles_none","vehicles_1","vehicles_2","vehicles_3plus",
    "total_households","avg_household_size","total_families","avg_family_size",
    "enroll_nursery_preschool","enroll_k12","enroll_kindergarten",
    "enroll_elem_grade1to4","enroll_elem_grade5to8",
    "enroll_hs_grade9to12","enroll_college_undergrad","enroll_grad_professional",
    "edu_9to12_no_diploma","edu_associates","edu_bachelors",
    "edu_bachelors_or_higher","edu_grad_professional",
    "edu_hs_graduate","edu_hs_or_higher","edu_less_than_9th","edu_less_than_hs",
    "edu_25to34_bachelors_or_higher","edu_25to34_hs_or_higher",
    "edu_some_college_or_associates","edu_some_college_no_degree",
    "poverty_male","poverty_female"]
ALL_POI_FEATURE_NAMES = [
    "poi_finance","poi_public","poi_transport","poi_entertainment","poi_health",
    "poi_service","poi_education","poi_government","poi_religion",
    "poi_accommodation","poi_food","poi_cafe","poi_fast_food","poi_ice_cream",
    "poi_pub","poi_restaurant","poi_shop_beauty","poi_shop_clothes",
    "poi_boutique","poi_shop_transport","poi_retail","poi_commodity",
    "poi_marketplace","poi_home_improvement","poi_sport","poi_public_transport",
    "poi_kindergarten","poi_office","poi_recycling","poi_travel_agency",
    "poi_tourism","poi_shop_livelihood","poi_residential","poi_dormitory"]
ALL_LU_FEATURE_NAMES = ["residential","business","recreation","industrial","transport","special","agriculture"]
ALL_JOBS_FEATURE_NAMES = ["jobs_total"]
assert len(ALL_DEMO_FEATURE_NAMES)==97; assert len(ALL_POI_FEATURE_NAMES)==34
SELECTED_DEMO=["pop_total","pop_male","pop_female","median_age_total","median_age_male","median_age_female","mean_travel_time_min"]
SELECTED_POI=ALL_POI_FEATURE_NAMES; SELECTED_LU=ALL_LU_FEATURE_NAMES; SELECTED_JOBS=ALL_JOBS_FEATURE_NAMES
DEMO_COL_IDX=[ALL_DEMO_FEATURE_NAMES.index(n) for n in SELECTED_DEMO]
POI_COL_IDX=[ALL_POI_FEATURE_NAMES.index(n) for n in SELECTED_POI]
LU_COL_IDX=[ALL_LU_FEATURE_NAMES.index(n) for n in SELECTED_LU]
JOBS_COL_IDX=[ALL_JOBS_FEATURE_NAMES.index(n) for n in SELECTED_JOBS]
FEATURE_NAMES=SELECTED_DEMO+SELECTED_POI
if USE_LU_FEATURES: FEATURE_NAMES+=SELECTED_LU
if USE_JOBS_FEATURES: FEATURE_NAMES+=SELECTED_JOBS
print(f"Features: {len(FEATURE_NAMES)}")


## Утилиты

In [ ]:
def precompute_coords(data_path, shp_path):
    for aid in os.listdir(data_path):
        ad=os.path.join(data_path,aid)
        if not os.path.isdir(ad): continue
        cp=os.path.join(ad,"coords.npy")
        if os.path.exists(cp): continue
        sf=os.path.join(shp_path,aid,f"{aid}.shp")
        if not os.path.exists(sf): continue
        try:
            gdf=gpd.read_file(sf).to_crs("EPSG:3857"); c=gdf.geometry.centroid
            np.save(cp,np.column_stack([c.x.values,c.y.values]))
        except: pass

def load_area(area_id):
    ap=os.path.join(DATA_PATH,area_id)
    try:
        fp=[np.load(os.path.join(ap,"demos.npy"))[:,DEMO_COL_IDX],np.load(os.path.join(ap,"pois.npy"))[:,POI_COL_IDX]]
        if USE_LU_FEATURES:
            lp=os.path.join(ap,"lu.npy")
            if os.path.exists(lp): fp.append(np.load(lp)[:,LU_COL_IDX])
        if USE_JOBS_FEATURES:
            jp=os.path.join(ap,"jobs.npy")
            if os.path.exists(jp): fp.append(np.load(jp)[:,JOBS_COL_IDX])
        nf=np.concatenate(fp,axis=1); adj=np.load(os.path.join(ap,"adj.npy"))
        dis=np.load(os.path.join(ap,"dis.npy")); od=np.load(os.path.join(ap,"od.npy"))
        cp=os.path.join(ap,"coords.npy"); co=np.load(cp) if os.path.exists(cp) else None
        return nf,adj,dis,od,co
    except: return None

def build_graph(adjacency,nfs,ds,dev,pe_type='rwpe'):
    ri,ci=np.where(adjacency>0)
    gd=Data(x=torch.FloatTensor(nfs),edge_index=torch.LongTensor(np.stack([ri,ci])),
        edge_attr=torch.FloatTensor(ds[ri,ci]).unsqueeze(-1),num_nodes=nfs.shape[0])
    if pe_type=='rwpe': gd=T.AddRandomWalkPE(walk_length=PE_WALK_LEN,attr_name='pe')(gd)
    elif pe_type=='spe': gd=T.AddLaplacianEigenvectorPE(k=PE_WALK_LEN,attr_name='pe')(gd)
    elif pe_type=='rrwp':
        gd=T.AddRandomWalkPE(walk_length=PE_WALK_LEN,attr_name='pe')(gd)
        N=nfs.shape[0]; a=torch.zeros(N,N); a[gd.edge_index[0],gd.edge_index[1]]=1.0
        d=a.sum(1).clamp(min=1); rw=a/d.unsqueeze(1); feats=[]; rk=rw.clone()
        for k in range(PE_WALK_LEN):
            feats.append(rk[gd.edge_index[0],gd.edge_index[1]].unsqueeze(-1))
            if k<PE_WALK_LEN-1: rk=rk@rw
        gd.rrwp_edge=torch.cat(feats,dim=-1)
    else: raise ValueError(f"Unknown pe_type: {pe_type}")
    return gd.to(dev)

def build_huber_weight_table(od,fm,bw=2.0,mp=1e-4):
    fl=od[fm].ravel(); fl=fl[fl>0].astype(float)
    if len(fl)<10: return None,None
    kde=gaussian_kde(fl,bw_method=bw/(fl.std()+1e-8))
    fg=np.linspace(0,fl.max()*1.05,2000)
    return fg,1.0/np.maximum(kde(np.sqrt(np.maximum(fg,0))),mp)

def interpolate_huber_weights(fv,fg,wt):
    if fg is None: return np.ones_like(fv,dtype=np.float32)
    w=np.interp(fv,fg,wt); w[fv==0]=1.0; return w.astype(np.float32)

def build_dest_dict(od):
    d={}
    for o,dest in zip(*np.where(od>0)): d.setdefault(int(o),[]).append(int(dest))
    return {k:np.array(v) for k,v in d.items()}

precompute_coords(DATA_PATH,SHP_PATH)


## Модели

In [ ]:
class GraphNormLayer(nn.Module):
    def __init__(self,hd):
        super().__init__(); self.gamma=nn.Parameter(torch.ones(hd)); self.beta=nn.Parameter(torch.zeros(hd))
        self.alpha=nn.Parameter(torch.ones(1)*0.5)
    def forward(self,x,batch=None):
        if batch is None: batch=torch.zeros(x.size(0),dtype=torch.long,device=x.device)
        mn=torch.zeros(batch.max()+1,x.size(1),device=x.device)
        ct=torch.zeros(batch.max()+1,1,device=x.device)
        mn.scatter_add_(0,batch.unsqueeze(1).expand_as(x),x)
        ct.scatter_add_(0,batch.unsqueeze(1)[:,:1],torch.ones(x.size(0),1,device=x.device))
        ct=ct.clamp(min=1); mn=mn/ct; xs=x-self.alpha*mn[batch]
        v=torch.zeros(batch.max()+1,x.size(1),device=x.device)
        v.scatter_add_(0,batch.unsqueeze(1).expand_as(xs),xs**2); v=v/ct
        return self.gamma*xs/(v[batch].sqrt()+1e-5)+self.beta

class GRANOLANorm(nn.Module):
    def __init__(self,hd):
        super().__init__(); self.bn=nn.LayerNorm(hd)
        self.pg=Sequential(Linear(hd,hd),ReLU(),Linear(hd,hd*2))
    def forward(self,x,batch=None):
        xn=self.bn(x); p=self.pg(x.detach()); g,b=p.chunk(2,dim=-1)
        return (1.0+g)*xn+b

class GPSEncoder(nn.Module):
    def __init__(self,idim,hd,ped,ed,nl,nh=4,do=0.1,pe_type='rwpe',norm_type='batch_norm'):
        super().__init__(); self.pe_type=pe_type; self.norm_type=norm_type
        npd=hd-ped; self.node_proj=Sequential(Linear(idim,npd),ReLU(),Linear(npd,npd))
        pid=PE_WALK_LEN
        if pe_type=='spe':
            self.pe_abs_proj=Sequential(Linear(pid,ped),ReLU(),Linear(ped,ped))
            self.pe_norm=BatchNorm1d(pid)
        else: self.pe_norm=BatchNorm1d(pid); self.pe_proj=Linear(pid,ped)
        if pe_type=='rrwp': self.rrwp_proj=Sequential(Linear(PE_WALK_LEN,hd),ReLU(),Linear(hd,nh))
        self.edge_proj=Sequential(Linear(ed,hd),ReLU(),Linear(hd,hd))
        self.gps_layers=ModuleList()
        for _ in range(nl):
            gmlp=Sequential(Linear(hd,hd),ReLU(),Linear(hd,hd))
            gn=norm_type if norm_type=='batch_norm' else None
            self.gps_layers.append(GPSConv(hd,GINEConv(gmlp),heads=nh,attn_type='multihead',norm=gn,attn_kwargs={'dropout':do}))
        if norm_type in ('graph_norm','granola'):
            NC=GraphNormLayer if norm_type=='graph_norm' else GRANOLANorm
            self.extra_norms=ModuleList([NC(hd) for _ in range(nl)])
        else: self.extra_norms=None

    def forward(self,gd):
        ne=self.node_proj(gd.x); pr=gd.pe
        if self.pe_type=='spe': pe=self.pe_abs_proj(torch.abs(self.pe_norm(pr)))
        else: pe=self.pe_proj(self.pe_norm(pr))
        h=torch.cat([ne,pe],dim=-1); ee=self.edge_proj(gd.edge_attr)
        batch=torch.zeros(gd.x.size(0),dtype=torch.long,device=gd.x.device)
        for i,l in enumerate(self.gps_layers):
            h=l(h,gd.edge_index,batch,edge_attr=ee)
            if self.extra_norms is not None: h=self.extra_norms[i](h,batch)
        return h

class BilinearDecoder(nn.Module):
    def __init__(self,hd): super().__init__(); self.W=nn.Parameter(torch.randn(hd,hd)*0.01)
    def forward(self,oe,de,d=None): return (oe*(de@self.W.T)).sum(-1)

class TransFlowerDecoder(nn.Module):
    def __init__(self,hd,nh=4,nl=2,do=0.1):
        super().__init__()
        self.fp=Sequential(Linear(hd*2+1,hd),ReLU(),Linear(hd,hd))
        tl=nn.TransformerEncoderLayer(d_model=hd,nhead=nh,dim_feedforward=hd*4,dropout=do,batch_first=True)
        self.tf=nn.TransformerEncoder(tl,num_layers=nl)
        self.ph=Sequential(Linear(hd,hd//2),ReLU(),Linear(hd//2,1))
    def forward(self,oe,de,d):
        fe=self.fp(torch.cat([oe,de,d],dim=-1))
        return self.ph(self.tf(fe.unsqueeze(0)).squeeze(0)).squeeze(-1)

class GPSODModel(nn.Module):
    def __init__(self,idim,hd,ped,ed,gl,gh,gdo,dt='bilinear',th=4,tl=2,tdo=0.1,pe_type='rwpe',nt='batch_norm'):
        super().__init__()
        self.encoder=GPSEncoder(idim,hd,ped,ed,gl,gh,gdo,pe_type=pe_type,norm_type=nt)
        self.decoder_type=dt; self.hidden_dim=hd
        if dt=='bilinear': self.decoder=BilinearDecoder(hd)
        elif dt=='transflower': self.decoder=TransFlowerDecoder(hd,th,tl,tdo)
        else: raise ValueError(dt)
        self.outflow_head=Linear(hd,1); self.inflow_head=Linear(hd,1)
    def encode(self,gd): return self.encoder(gd)
    def decode_row(self,ne,oi,di,dm):
        D=di.size(0); return self.decoder(ne[oi].unsqueeze(0).expand(D,-1),ne[di],dm[oi,di].unsqueeze(-1))
    def predict_node_flows(self,ne): return self.outflow_head(ne).squeeze(-1),self.inflow_head(ne).squeeze(-1)

def make_model(config,input_dim=None,edge_dim=None,graph_data_ref=None):
    if input_dim is None and graph_data_ref is not None: input_dim=graph_data_ref.x.size(-1)
    if edge_dim is None and graph_data_ref is not None: edge_dim=graph_data_ref.edge_attr.size(-1)
    assert input_dim and edge_dim
    return GPSODModel(input_dim,HIDDEN_DIM,PE_DIM,edge_dim,GPS_LAYERS,GPS_HEADS,GPS_DROPOUT,
        config.decoder_type,TF_HEADS,TF_LAYERS,TF_DROPOUT,config.pe_type,config.gps_norm_type).to(device)


## Семплирование и Loss

In [ ]:
def sample_destinations(oi,nzd,nn,use_sampling=True,n_dest=128,inc_zeros=True,zr=0.3):
    nz=nzd.get(oi,np.array([],dtype=int))
    if not use_sampling: return np.arange(nn)
    if not inc_zeros:
        av=nz if len(nz)>0 else np.arange(nn)
        return av if len(av)<=n_dest else np.random.choice(av,n_dest,replace=False)
    nzn=max(1,int(n_dest*zr)); nnz=n_dest-nzn
    snz=nz if len(nz)<=nnz else np.random.choice(nz,nnz,replace=False)
    zd=np.setdiff1d(np.arange(nn),nz)
    sz=np.random.choice(zd,min(nzn,len(zd)),replace=False) if len(zd)>0 else np.array([],dtype=int)
    return np.concatenate([snz,sz]).astype(int)

def compute_loss_for_city(model,cd,config,origin_batch_indices=None):
    lt=config.loss_type; pm=config.prediction_mode; ul=config.use_log_transform
    ne=model.encode(cd['graph_data']); nn_=cd['num_nodes']; od=cd['od_matrix_train']
    of=cd['outflow_train']; inf_=cd['inflow_train']; nzd=cd['nonzero_dest_dict']
    ao=list(nzd.keys()) if origin_batch_indices is None else origin_batch_indices
    tl=torch.tensor(0.0,requires_grad=True,device=device); np_=0
    for oi in ao:
        di=sample_destinations(oi,nzd,nn_,config.use_dest_sampling,config.n_dest_sample,config.include_zero_pairs,config.zero_pair_ratio)
        if len(di)==0: continue
        dt=torch.LongTensor(di).to(device); rf=od[oi,di].astype(float)
        tr=np.log1p(rf) if ul and lt!='ce' else rf.copy()
        if pm=='normalized':
            oof=of[oi]
            if oof<1: continue
            tv=tr/(np.log1p(oof)+1e-8) if ul and lt!='ce' else tr/(oof+1e-8)
        else: tv=tr
        tt=torch.FloatTensor(tv).to(device)
        sc=model.decode_row(ne,oi,dt,cd['distance_matrix'])
        if lt=='huber':
            pr=F.softmax(sc,dim=0) if pm=='normalized' else F.relu(sc)
            w=torch.FloatTensor(interpolate_huber_weights(rf,cd['huber_flow_grid'],cd['huber_weight_table'])).to(device)
            df=torch.abs(pr-tt); h=torch.where(df<=HUBER_DELTA,0.5*df**2,HUBER_DELTA*df-0.5*HUBER_DELTA**2)
            rl=(w*h).mean()
        elif lt=='ce':
            p=F.softmax(sc,dim=0); ct=torch.FloatTensor(rf/(of[oi]+1e-8)).to(device)
            rl=-torch.sum(ct*torch.log(p+1e-10))
        elif lt=='multitask':
            pr=F.softmax(sc,dim=0) if pm=='normalized' else F.relu(sc); rl=F.mse_loss(pr,tt)
        elif lt=='zinb':
            mu=F.softplus(sc)+1e-4; th=torch.ones_like(mu)*10.0; pi=torch.sigmoid(sc*0.1)
            ft=torch.FloatTensor(rf).to(device); iz=(ft<0.5).float(); tc=th.clamp(min=1e-4)
            nb=(torch.lgamma(ft+tc)-torch.lgamma(tc)-torch.lgamma(ft+1)+tc*torch.log(tc/(tc+mu))+ft*torch.log(mu/(tc+mu)))
            nz=torch.exp(tc*torch.log(tc/(tc+mu)))
            lp=iz*torch.log(pi+(1-pi)*nz+1e-10)+(1-iz)*(torch.log(1-pi+1e-10)+nb); rl=-lp.mean()
        else: raise ValueError(lt)
        tl=tl+rl; np_+=1
    if np_==0: return torch.tensor(0.0,requires_grad=True,device=device)
    ml=tl/np_
    if lt=='multitask':
        po,pi_=model.predict_node_flows(ne)
        if pm=='normalized':
            tf_=float(od.sum())+1e-8; to_=torch.FloatTensor(of/tf_).to(device); ti_=torch.FloatTensor(inf_/tf_).to(device)
        else: to_=torch.FloatTensor(of).to(device); ti_=torch.FloatTensor(inf_).to(device)
        ol=F.mse_loss(po,to_); il=F.mse_loss(pi_,ti_)
        if NORMALIZE_MULTITASK:
            nzf=od[od>0].astype(float); tva=nzf/(od.sum()+1e-8) if pm=='normalized' else nzf
            dm=(torch.FloatTensor(tva).to(device)**2).mean()+1e-8
            do_=(to_**2).mean()+1e-8; di_=(ti_**2).mean()+1e-8
            ml=ml/dm; ol=ol/do_; il=il/di_
        return LAMBDA_MAIN*ml+(LAMBDA_SUB/2)*(ol+il)
    return ml


## Метрики и предсказания

In [ ]:
def compute_cpc(p,t):
    d=p.sum()+t.sum(); return 0.0 if d<1e-10 else float(2*np.minimum(p,t).sum()/d)

def compute_metrics(p,t):
    return {'CPC':compute_cpc(p,t),'MAE':float(np.abs(p-t).mean()),'RMSE':float(np.sqrt(((p-t)**2).mean()))}

def predict_full_matrix(model,cd,config,dbs=256):
    model.eval(); pm=config.prediction_mode; ul=config.use_log_transform
    nn_=cd['num_nodes']; of=cd['outflow_full']; pred=np.zeros((nn_,nn_),dtype=np.float32)
    with torch.no_grad():
        ne=model.encode(cd['graph_data'])
        for oi in range(nn_):
            row=np.zeros(nn_,dtype=np.float32)
            for bs in range(0,nn_,dbs):
                be=min(bs+dbs,nn_); di=torch.LongTensor(np.arange(bs,be)).to(device)
                row[bs:be]=model.decode_row(ne,oi,di,cd['distance_matrix']).cpu().numpy()
            if config.loss_type=='zinb': row=np.log1p(np.exp(row))
            elif pm=='normalized': row=np.exp(row-row.max()); row=row/(row.sum()+1e-10)*of[oi]
            else: row=np.maximum(row,0)
            if ul and config.loss_type not in ('ce','zinb'): row=np.expm1(np.maximum(row,0))
            pred[oi]=row
    return pred

def evaluate_full_matrix(model,cd,config,dbs=256):
    pred=predict_full_matrix(model,cd,config,dbs); od=cd['od_matrix_np']; nz=od>0
    return pred,compute_metrics(pred.ravel(),od.ravel().astype(float)),compute_metrics(pred[nz],od[nz].astype(float))


## Загрузка данных

In [ ]:
data=load_area(SINGLE_CITY_ID)
assert data is not None
node_features_raw,adjacency,distances,od_matrix_raw,coords=data
num_nodes=node_features_raw.shape[0]; FEAT_DIM=node_features_raw.shape[1]
np.random.seed(42); nzo,nzd_=np.where(od_matrix_raw>0); np_=len(nzo); perm=np.random.permutation(np_)
nt=int(np_*0.8); nv=int(np_*0.9)
train_mask=np.zeros((num_nodes,num_nodes),bool); val_mask=np.zeros((num_nodes,num_nodes),bool); test_mask=np.zeros((num_nodes,num_nodes),bool)
train_mask[nzo[perm[:nt]],nzd_[perm[:nt]]]=True; val_mask[nzo[perm[nt:nv]],nzd_[perm[nt:nv]]]=True; test_mask[nzo[perm[nv:]],nzd_[perm[nv:]]]=True
toi=np.where(train_mask.any(1))[0]; nfs=MinMaxScaler().fit(node_features_raw[toi]).transform(node_features_raw)
node_features_scaled=nfs; ds=MinMaxScaler().fit(distances[train_mask].reshape(-1,1)).transform(distances.reshape(-1,1)).reshape(num_nodes,num_nodes)
distances_scaled=ds; od_train=od_matrix_raw*train_mask; od_val=od_matrix_raw*val_mask
outflow_full=od_matrix_raw.sum(1); inflow_full=od_matrix_raw.sum(0)
outflow_train=od_train.sum(1); inflow_train=od_train.sum(0); outflow_val=od_val.sum(1); inflow_val=od_val.sum(0)
train_dest_dict=build_dest_dict(od_train); val_dest_dict=build_dest_dict(od_val)
huber_flow_grid,huber_weight_table=build_huber_weight_table(od_matrix_raw,train_mask,HUBER_KDE_BW,HUBER_MIN_PROB)
print(f"{SINGLE_CITY_ID}: N={num_nodes}, feat={FEAT_DIM}, train:{train_mask.sum():,} val:{val_mask.sum():,} test:{test_mask.sum():,}")


In [ ]:
def build_single_city_data(pe_type='rwpe'):
    gd=build_graph(adjacency,node_features_scaled,distances_scaled,device,pe_type=pe_type)
    dt=torch.FloatTensor(distances_scaled).to(device)
    return {'city_id':SINGLE_CITY_ID,'graph_data':gd,'distance_matrix':dt,
        'od_matrix_np':od_matrix_raw,'od_matrix_train':od_train,
        'outflow_full':outflow_full,'inflow_full':inflow_full,
        'outflow_train':outflow_train,'inflow_train':inflow_train,
        'outflow_val':outflow_val,'inflow_val':inflow_val,
        'num_nodes':num_nodes,'nonzero_dest_dict':train_dest_dict,
        'huber_flow_grid':huber_flow_grid,'huber_weight_table':huber_weight_table,
        'train_mask':train_mask,'val_mask':val_mask,'test_mask':test_mask,
        'val_dest_dict':val_dest_dict,'node_features_scaled':node_features_scaled,'distances_scaled':distances_scaled}

_scd_cache={}
def get_single_city_data(pe_type='rwpe'):
    if pe_type not in _scd_cache:
        print(f"  Building graph pe_type={pe_type}..."); _scd_cache[pe_type]=build_single_city_data(pe_type)
    return _scd_cache[pe_type]
single_city_data=get_single_city_data('rwpe')


## Единый цикл обучения (single-city и multi-city)

In [ ]:
def _train_loop(run_id, run_name, config, model, city_datas, is_multi=False):
    """
    Единый цикл обучения для single-city и multi-city.
    city_datas: dict {city_id: city_data} — для single = один город, для multi = несколько.
    """
    print(f"\n{'='*70}\n  {run_name}\n  {config.describe()}\n{'='*70}")
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-5)

    max_epochs = config.mc_epochs if is_multi else config.epochs
    history = {'train_loss':[], 'val_loss':[], 'val_cpc_full':[], 'val_cpc_nz':[]}
    best_val_loss = float('inf'); patience_count = 0; best_state = None

    # Разделяем города на train / val
    if is_multi:
        train_cds = {cid: city_datas[cid] for cid in train_city_ids if cid in city_datas}
        val_cds   = {cid: city_datas[cid] for cid in val_city_ids if cid in city_datas}
    else:
        # Single-city: один город, val через val_mask
        cid = list(city_datas.keys())[0]; cd = city_datas[cid]
        train_cds = {cid: cd}
        val_cd = dict(cd)
        val_cd['od_matrix_train'] = cd['od_matrix_np'] * cd['val_mask']
        val_cd['outflow_train'] = cd['outflow_val']
        val_cd['inflow_train'] = cd['inflow_val']
        val_cd['nonzero_dest_dict'] = cd['val_dest_dict']
        val_cds = {cid: val_cd}

    epoch = 0
    status = 'ok'
    for epoch in range(1, max_epochs + 1):
        model.train(); t0 = time.time()
        city_ids_shuffled = list(train_cds.keys()); np.random.shuffle(city_ids_shuffled)
        epoch_losses = []; nan_count = 0; total_batches = 0

        for cid in city_ids_shuffled:
            cd = train_cds[cid]
            cc = replace(config, n_dest_sample=cd.get('city_n_dest', config.n_dest_sample)) if is_multi else config
            origins = np.array(list(cd['nonzero_dest_dict'].keys())); np.random.shuffle(origins)
            for bs in range(0, len(origins), ORIGIN_BATCH_SIZE):
                batch = origins[bs:bs+ORIGIN_BATCH_SIZE].tolist()
                optimizer.zero_grad()
                loss = compute_loss_for_city(model, cd, cc, origin_batch_indices=batch)
                total_batches += 1
                # NaN protection
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1; continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_losses.append(loss.item())

        # Проверка дивергенции
        if total_batches > 0 and nan_count / total_batches > NAN_BATCH_THRESHOLD:
            print(f"  ✗ NaN divergence @ epoch {epoch} ({nan_count}/{total_batches} NaN batches)")
            status = 'nan_diverged'; break

        train_loss = np.mean(epoch_losses) if epoch_losses else float('nan')

        # ── Валидация: loss + CPC по полной матрице ──────────────────────
        model.eval()
        val_losses = []; val_cpc_fulls = []; val_cpc_nzs = []
        with torch.no_grad():
            for cid, vcd in val_cds.items():
                vl = compute_loss_for_city(model, vcd, config)
                if not (torch.isnan(vl) or torch.isinf(vl)):
                    val_losses.append(vl.item())
                # CPC по полной матрице (единый подход!)
                _, mf, mnz = evaluate_full_matrix(model, city_datas[cid] if is_multi else list(city_datas.values())[0], config, dest_batch_size=DEST_BATCH_SIZE)
                val_cpc_fulls.append(mf['CPC']); val_cpc_nzs.append(mnz['CPC'])

        avg_val_loss = np.mean(val_losses) if val_losses else float('nan')
        avg_cpc_full = np.mean(val_cpc_fulls); avg_cpc_nz = np.mean(val_cpc_nzs)
        history['train_loss'].append(train_loss); history['val_loss'].append(avg_val_loss)
        history['val_cpc_full'].append(avg_cpc_full); history['val_cpc_nz'].append(avg_cpc_nz)

        if not np.isnan(avg_val_loss): scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0; flag = ' ✓'
        else: patience_count += 1; flag = ''

        if epoch % 5 == 0 or epoch == 1:
            nan_str = f" NaN:{nan_count}" if nan_count > 0 else ""
            print(f"  {epoch:3d}/{max_epochs}  train={train_loss:.4f}  val={avg_val_loss:.4f}  "
                  f"CPC_full={avg_cpc_full:.4f}  CPC_nz={avg_cpc_nz:.4f}  "
                  f"{time.time()-t0:.1f}s{flag}{nan_str}")
        if patience_count >= config.patience:
            print(f"  Early stop @ epoch {epoch}"); break

    if status == 'nan_diverged':
        dummy = {'CPC': 0.0, 'MAE': float('inf'), 'RMSE': float('inf')}
        save_metrics_to_csv(run_id, run_name, config, dummy, dummy, dummy, n_params, epoch, status)
        return {'name': run_name, 'model': model, 'config': config, 'history': history,
                'metrics_full': dummy, 'metrics_nonzero': dummy, 'metrics_test_pairs': dummy,
                'pred_matrix': None, 'status': status}

    # ── Оценка на последних и лучших весах ───────────────────────────────
    eval_cities = city_datas if is_multi else {list(city_datas.keys())[0]: list(city_datas.values())[0]}

    def eval_all(label):
        all_mf = []; all_mnz = []; all_mt = []; per_city = []
        for cid, cd in eval_cities.items():
            pred, mf, mnz = evaluate_full_matrix(model, cd, config, dest_batch_size=DEST_BATCH_SIZE)
            mt = compute_metrics(pred[cd['test_mask']], cd['od_matrix_np'][cd['test_mask']].astype(float))
            all_mf.append(mf); all_mnz.append(mnz); all_mt.append(mt)
            per_city.append({'city_id': cid, 'CPC_full': mf['CPC'], 'CPC_nz': mnz['CPC'],
                             'MAE': mf['MAE'], 'RMSE': mf['RMSE'], 'CPC_test': mt['CPC']})
        avg_mf = {k: np.mean([m[k] for m in all_mf]) for k in all_mf[0]}
        avg_mnz = {k: np.mean([m[k] for m in all_mnz]) for k in all_mnz[0]}
        avg_mt = {k: np.mean([m[k] for m in all_mt]) for k in all_mt[0]}
        print(f"\n  === {label} ===")
        for pc in per_city:
            print(f"    {pc['city_id']}: CPC_full={pc['CPC_full']:.4f}  CPC_nz={pc['CPC_nz']:.4f}  CPC_test={pc['CPC_test']:.4f}")
        print(f"  Avg: CPC_full={avg_mf['CPC']:.4f}  CPC_nz={avg_mnz['CPC']:.4f}  CPC_test={avg_mt['CPC']:.4f}  MAE={avg_mf['MAE']:.4f}")
        return avg_mf, avg_mnz, avg_mt, per_city

    mf_last, mnz_last, mt_last, pc_last = eval_all(f"Последние веса (эпоха {epoch})")

    if best_state: model.load_state_dict(best_state)
    mf_best, mnz_best, mt_best, pc_best = eval_all("Лучшие веса (best val_loss)")

    if mnz_last['CPC'] > mnz_best['CPC']:
        print(f"\n  ⚠ Последние веса лучше по CPC_nz! ({mnz_last['CPC']:.4f} > {mnz_best['CPC']:.4f})")
        mf, mnz, mt, pc = mf_last, mnz_last, mt_last, pc_last
    else:
        mf, mnz, mt, pc = mf_best, mnz_best, mt_best, pc_best

    # ── Сохранение ───────────────────────────────────────────────────────
    save_metrics_to_csv(run_id, run_name, config, mf, mnz, mt, n_params, epoch, status)
    save_model_weights(run_id, model)

    return {'name': run_name, 'model': model, 'config': config, 'history': history,
            'metrics_full': mf, 'metrics_nonzero': mnz, 'metrics_test_pairs': mt,
            'per_city': pc, 'status': status}


def train_single_city(run_id, run_name, config):
    cd = get_single_city_data(config.pe_type)
    model = make_model(config, graph_data_ref=cd['graph_data'])
    return _train_loop(run_id, run_name, config, model, {SINGLE_CITY_ID: cd}, is_multi=False)


def train_multi_city(run_id, run_name, config, city_data_dict):
    input_dim = city_data_dict[list(city_data_dict.keys())[0]]['graph_data'].x.shape[1]
    model = make_model(config, input_dim=input_dim, edge_dim=1)
    return _train_loop(run_id, run_name, config, model, city_data_dict, is_multi=True)


## LGBM поверх GPS-эмбеддингов

In [ ]:
def train_lgbm_from_model(run_id, city_data, donor_model, donor_name):
    """Обучает LightGBM на GPS-эмбеддингах от donor_model."""
    print(f"\n{'='*70}\n  LGBM: {run_id} (донор: {donor_name})\n{'='*70}")
    nn_ = city_data['num_nodes']; od = city_data['od_matrix_np']
    tm = city_data['train_mask']; vm = city_data['val_mask']; tsm = city_data['test_mask']
    nfs = city_data['node_features_scaled']; ds = city_data['distances_scaled']

    donor_model.eval()
    with torch.no_grad():
        embs = donor_model.encode(city_data['graph_data']).cpu().numpy()
    ed = embs.shape[1]; nfd = nfs.shape[1]

    def build_features(mask):
        oi, di = np.where(mask); n = len(oi)
        feat = np.zeros((n, ed*2+1+nfd*2), dtype=np.float32)
        feat[:, :ed] = embs[oi]; feat[:, ed:2*ed] = embs[di]
        feat[:, 2*ed] = ds[oi, di]
        feat[:, 2*ed+1:2*ed+1+nfd] = nfs[oi]; feat[:, 2*ed+1+nfd:] = nfs[di]
        return feat, od[oi, di].astype(float), oi, di

    X_train, y_train, _, _ = build_features(tm)
    X_val, y_val, _, _ = build_features(vm)
    print(f"  Train: {len(y_train):,} pairs, Val: {len(y_val):,} pairs")

    params = {'objective':'regression','metric':'mae','learning_rate':0.05,
              'num_leaves':63,'max_depth':8,'subsample':0.8,'colsample_bytree':0.8,'verbose':-1,'seed':42}
    lgbm_model = lgb.train(params, lgb.Dataset(X_train, y_train), num_boost_round=1000,
        valid_sets=[lgb.Dataset(X_val, y_val)], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)])

    # Предсказание полной матрицы
    print(f"  Predicting full {nn_}x{nn_} matrix...")
    full_mask = np.ones((nn_, nn_), bool)
    X_full, _, ao, ad = build_features(full_mask)
    pf = np.maximum(lgbm_model.predict(X_full), 0)
    pred = np.zeros((nn_, nn_), dtype=np.float32); pred[ao, ad] = pf.astype(np.float32)

    mf = compute_metrics(pred.ravel(), od.ravel().astype(float))
    nzm = od > 0; mnz = compute_metrics(pred[nzm], od[nzm].astype(float))
    mt = compute_metrics(pred[tsm], od[tsm].astype(float))

    print(f"  Full:    CPC={mf['CPC']:.4f}  MAE={mf['MAE']:.4f}")
    print(f"  Nonzero: CPC={mnz['CPC']:.4f}")
    print(f"  Test:    CPC={mt['CPC']:.4f}")

    # Сохранение в CSV
    dummy_config = TrainingConfig(decoder_type='lgbm', loss_type='mae')
    row = {'timestamp': datetime.now().isoformat(), 'run_id': run_id, 'name': f'LGBM({donor_name})',
           'status': 'ok', 'decoder': 'lgbm', 'loss_type': 'mae',
           'prediction_mode': 'raw', 'pe_type': '-', 'gps_norm_type': '-',
           'use_log_transform': False, 'n_params': 0, 'epochs_trained': lgbm_model.best_iteration,
           'CPC_full': mf['CPC'], 'CPC_nz': mnz['CPC'], 'CPC_test': mt['CPC'],
           'MAE_full': mf['MAE'], 'RMSE_full': mf['RMSE']}
    with open(METRICS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=row.keys()); w.writerow(row)

    return {'name': f'LGBM({donor_name})', 'model': lgbm_model, 'metrics_full': mf,
            'metrics_nonzero': mnz, 'metrics_test_pairs': mt, 'pred_matrix': pred,
            'config': dummy_config, 'status': 'ok'}


## Запуск single-city (Approach B)

In [ ]:
single_city_results = {}

single_city_runs = [
    ('B1','B1: GPS+Bilinear+Huber (raw)', TrainingConfig(decoder_type='bilinear',loss_type='huber',prediction_mode='raw')),
    ('B2','B2: GPS+TransFlower+Huber (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw')),
    ('B3','B3: GPS+Bilinear+Multitask (raw)', TrainingConfig(decoder_type='bilinear',loss_type='multitask',prediction_mode='raw')),
    ('B4','B4: GPS+TransFlower+Multitask (raw)', TrainingConfig(decoder_type='transflower',loss_type='multitask',prediction_mode='raw')),
    ('B6','B6: GPS+Bilinear+CE (normalized)', TrainingConfig(decoder_type='bilinear',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False)),
    ('B7','B7: GPS+TransFlower+CE (normalized)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False)),
    ('B2n','B2n: GPS+TransFlower+Huber (normalized)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='normalized')),
    # Huber одиночные
    ('B2+spe','B2+SPE: GPS+TF+Huber+SPE (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='spe')),
    ('B2+rrwp','B2+RRWP: GPS+TF+Huber+RRWP (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='rrwp')),
    ('B2+gnorm','B2+GraphNorm: GPS+TF+Huber+GraphNorm (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',gps_norm_type='graph_norm')),
    ('B2+granola','B2+GRANOLA: GPS+TF+Huber+GRANOLA (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',gps_norm_type='granola')),
    ('B2+zinb','B2+ZINB: GPS+TF+ZINB (raw)', TrainingConfig(decoder_type='transflower',loss_type='zinb',prediction_mode='raw',include_zero_pairs=True,zero_pair_ratio=0.5)),
    ('B2+log','B2+Log: GPS+TF+Huber+Log (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',use_log_transform=True)),
    # CE одиночные
    ('B7+spe','B7+SPE: GPS+TF+CE+SPE (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False,pe_type='spe')),
    ('B7+rrwp','B7+RRWP: GPS+TF+CE+RRWP (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False,pe_type='rrwp')),
    ('B7+gnorm','B7+GraphNorm: GPS+TF+CE+GraphNorm (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False,gps_norm_type='graph_norm')),
    ('B7+granola','B7+GRANOLA: GPS+TF+CE+GRANOLA (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False,gps_norm_type='granola')),
    # CE семплирование
    ('B7+samp','B7+samp128: GPS+TF+CE+samp128', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,n_dest_sample=128,include_zero_pairs=False)),
    ('B7+samp256','B7+samp256: GPS+TF+CE+samp256', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,n_dest_sample=256,include_zero_pairs=False)),
    ('B7+sz30','B7+sz30: GPS+TF+CE+samp+zeros30%', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,n_dest_sample=128,include_zero_pairs=True,zero_pair_ratio=0.3)),
    ('B7+sz50','B7+sz50: GPS+TF+CE+samp+zeros50%', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,n_dest_sample=128,include_zero_pairs=True,zero_pair_ratio=0.5)),
    # Комбинации
    ('B2+combo1','B2+SPE+GraphNorm: GPS+TF+Huber+SPE+GN (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='spe',gps_norm_type='graph_norm')),
    ('B2+combo2','B2+SPE+ZINB: GPS+TF+ZINB+SPE (raw)', TrainingConfig(decoder_type='transflower',loss_type='zinb',prediction_mode='raw',pe_type='spe',include_zero_pairs=True,zero_pair_ratio=0.5)),
    ('B7+combo1','B7+SPE+GraphNorm: GPS+TF+CE+SPE+GN (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=False,pe_type='spe',gps_norm_type='graph_norm')),
]

RUNS_HUBER=['B2','B2+spe','B2+rrwp','B2+gnorm','B2+granola','B2+zinb','B2+log']
RUNS_CE=['B7','B7+spe','B7+rrwp','B7+gnorm','B7+granola']
RUNS_CE_SAMP=['B7+samp','B7+samp256','B7+sz30','B7+sz50']
RUNS_COMBOS=['B2+combo1','B2+combo2','B7+combo1']
RUNS_TO_EXECUTE = RUNS_HUBER + RUNS_CE + RUNS_CE_SAMP + RUNS_COMBOS

for rid, rname, rcfg in single_city_runs:
    if rid not in RUNS_TO_EXECUTE: continue
    single_city_results[rid] = train_single_city(rid, rname, rcfg)


## LGBM с разными донорами

In [ ]:
lgbm_results = {}

# Перебираем все успешно обученные single-city модели как доноров
for donor_id, donor_result in single_city_results.items():
    if donor_result.get('status') != 'ok': continue
    if donor_result.get('model') is None: continue
    lgbm_id = f"L_{donor_id}"
    lgbm_results[lgbm_id] = train_lgbm_from_model(
        lgbm_id, single_city_data, donor_result['model'], donor_id
    )


## Multi-city (Approach C)

In [ ]:
print(f"Loading {len(MULTI_CITY_IDS)} cities...")
multi_city_raw = {}
for cid in MULTI_CITY_IDS:
    data = load_area(cid)
    if data is None: print(f"  [SKIP] {cid}"); continue
    nf,adj,dis,od,co = data
    if nf.shape[0]<10: print(f"  [SKIP] {cid}: too few"); continue
    multi_city_raw[cid] = {'nfeat':nf,'adj':adj,'dis':dis,'od':od,'coords':co}
    print(f"  {cid}: N={nf.shape[0]}")
COMMON_FEAT_DIM = min(v['nfeat'].shape[1] for v in multi_city_raw.values())
for cid,raw in multi_city_raw.items():
    nf=raw['nfeat'][:,:COMMON_FEAT_DIM]; dis=raw['dis']
    raw['nfeat_scaled']=MinMaxScaler().fit(nf).transform(nf)
    raw['dis_scaled']=MinMaxScaler().fit(dis.reshape(-1,1)).transform(dis.reshape(-1,1)).reshape(nf.shape[0],nf.shape[0])
mc_city_ids=list(multi_city_raw.keys()); np.random.seed(42); np.random.shuffle(mc_city_ids)
train_city_ids=mc_city_ids[:8]; val_city_ids=mc_city_ids[8:9]; test_city_ids=mc_city_ids[9:]
print(f"Train:{train_city_ids}\nVal:{val_city_ids}\nTest:{test_city_ids}")


In [ ]:
def prepare_city_data(cid,raw,pe_type='rwpe'):
    nfs=raw['nfeat_scaled']; ds=raw['dis_scaled']; od=raw['od']; nn_=nfs.shape[0]
    g=build_graph(raw['adj'],nfs,ds,device,pe_type); dt=torch.FloatTensor(ds).to(device)
    nzd=build_dest_dict(od); hfg,hwt=build_huber_weight_table(od,od>0,HUBER_KDE_BW,HUBER_MIN_PROB)
    cnd=max(8,min(int(np.mean([len(v) for v in nzd.values()])),128)) if nzd else N_DEST_SAMPLE
    return {'city_id':cid,'graph_data':g,'distance_matrix':dt,'od_matrix_np':od,'od_matrix_train':od,
        'outflow_full':od.sum(1),'outflow_train':od.sum(1),'inflow_train':od.sum(0),'inflow_full':od.sum(0),
        'outflow_val':od.sum(1),'inflow_val':od.sum(0),'num_nodes':nn_,'nonzero_dest_dict':nzd,
        'huber_flow_grid':hfg,'huber_weight_table':hwt,'city_n_dest':cnd,
        'train_mask':od>0,'val_mask':od>0,'test_mask':od>0,'val_dest_dict':nzd,
        'node_features_scaled':nfs,'distances_scaled':ds}

_mcd_cache={}
def get_mc_city_data(pe_type='rwpe'):
    if pe_type not in _mcd_cache:
        print(f"MC data pe_type={pe_type}..."); d={}
        for cid in mc_city_ids: d[cid]=prepare_city_data(cid,multi_city_raw[cid],pe_type); print(f"  {cid}: N={d[cid]['num_nodes']}")
        _mcd_cache[pe_type]=d
    return _mcd_cache[pe_type]


In [ ]:
mc_runs = [
    ('C1','C1: MC GPS+TF+Huber (raw)', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',mc_epochs=MC_EPOCHS)),
    ('C2','C2: MC GPS+TF+CE (norm)', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,mc_epochs=MC_EPOCHS)),
    ('C3','C3: MC GPS+TF+Multitask (raw)', TrainingConfig(decoder_type='transflower',loss_type='multitask',prediction_mode='raw',mc_epochs=MC_EPOCHS)),
    # Huber одиночные
    ('C1+spe','C1+SPE: MC GPS+TF+Huber+SPE', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='spe',mc_epochs=MC_EPOCHS)),
    ('C1+rrwp','C1+RRWP: MC GPS+TF+Huber+RRWP', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='rrwp',mc_epochs=MC_EPOCHS)),
    ('C1+gnorm','C1+GraphNorm: MC GPS+TF+Huber+GN', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',gps_norm_type='graph_norm',mc_epochs=MC_EPOCHS)),
    ('C1+granola','C1+GRANOLA: MC GPS+TF+Huber+GRANOLA', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',gps_norm_type='granola',mc_epochs=MC_EPOCHS)),
    ('C1+zinb','C1+ZINB: MC GPS+TF+ZINB', TrainingConfig(decoder_type='transflower',loss_type='zinb',prediction_mode='raw',include_zero_pairs=True,zero_pair_ratio=0.5,mc_epochs=MC_EPOCHS)),
    ('C1+log','C1+Log: MC GPS+TF+Huber+Log', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',use_log_transform=True,mc_epochs=MC_EPOCHS)),
    # CE одиночные
    ('C2+spe','C2+SPE: MC GPS+TF+CE+SPE', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,pe_type='spe',mc_epochs=MC_EPOCHS)),
    ('C2+rrwp','C2+RRWP: MC GPS+TF+CE+RRWP', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,pe_type='rrwp',mc_epochs=MC_EPOCHS)),
    ('C2+gnorm','C2+GraphNorm: MC GPS+TF+CE+GN', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,gps_norm_type='graph_norm',mc_epochs=MC_EPOCHS)),
    ('C2+granola','C2+GRANOLA: MC GPS+TF+CE+GRANOLA', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,gps_norm_type='granola',mc_epochs=MC_EPOCHS)),
    # Комбинации
    ('C1+combo1','C1+SPE+GN: MC GPS+TF+Huber+SPE+GN', TrainingConfig(decoder_type='transflower',loss_type='huber',prediction_mode='raw',pe_type='spe',gps_norm_type='graph_norm',mc_epochs=MC_EPOCHS)),
    ('C1+combo2','C1+SPE+ZINB: MC GPS+TF+ZINB+SPE', TrainingConfig(decoder_type='transflower',loss_type='zinb',prediction_mode='raw',pe_type='spe',include_zero_pairs=True,zero_pair_ratio=0.5,mc_epochs=MC_EPOCHS)),
    ('C2+combo1','C2+SPE+GN: MC GPS+TF+CE+SPE+GN', TrainingConfig(decoder_type='transflower',loss_type='ce',prediction_mode='normalized',use_dest_sampling=True,include_zero_pairs=False,pe_type='spe',gps_norm_type='graph_norm',mc_epochs=MC_EPOCHS)),
]

RUNS_MC_H=['C1','C1+spe','C1+rrwp','C1+gnorm','C1+granola','C1+zinb','C1+log']
RUNS_MC_CE=['C2','C2+spe','C2+rrwp','C2+gnorm','C2+granola']
RUNS_MC_COMBO=['C1+combo1','C1+combo2','C2+combo1']
MC_RUNS_TO_EXECUTE = RUNS_MC_H + RUNS_MC_CE + RUNS_MC_COMBO

multi_city_results = {}
for rid,rname,rcfg in mc_runs:
    if rid not in MC_RUNS_TO_EXECUTE: continue
    mcd = get_mc_city_data(rcfg.pe_type)
    multi_city_results[rid] = train_multi_city(rid, rname, rcfg, mcd)


## Сводные таблицы и графики

In [ ]:
def print_summary(rd, title):
    if not rd: print(f"  {title}: нет"); return
    print(f"\n{'='*130}\n  {title}\n{'='*130}")
    print(f"  {'ID':<14} {'Name':<50} {'Loss':<8} {'PE':<6} {'Norm':<12} {'CPC_f':>7} {'CPC_nz':>7} {'CPC_t':>7} {'MAE':>8} {'Status':<12}")
    bc=max((r['metrics_nonzero']['CPC'] for r in rd.values() if r.get('metrics_nonzero',{}).get('CPC',0)>0),default=0)
    for k in sorted(rd.keys()):
        r=rd[k]; mf=r.get('metrics_full',{}); mnz=r.get('metrics_nonzero',{}); mt=r.get('metrics_test_pairs',{})
        cfg=r.get('config',TrainingConfig()); st=r.get('status','ok')
        m='←' if abs(mnz.get('CPC',0)-bc)<1e-6 else ''
        print(f"  {k:<14} {r['name']:<50} {cfg.loss_type:<8} {cfg.pe_type:<6} {cfg.gps_norm_type:<12} "
              f"{mf.get('CPC',0):>7.4f} {mnz.get('CPC',0):>7.4f} {mt.get('CPC',0):>7.4f} {mf.get('MAE',0):>8.4f} {st:<12}{m}")
    print(f"{'='*130}")

print_summary(single_city_results, f"Single-city | {SINGLE_CITY_ID}")
print_summary(lgbm_results, "LGBM (single-city donors)")
print_summary(multi_city_results, "Multi-city")

# CSV итог
if METRICS_CSV.exists():
    print(f"\n  Все метрики сохранены в: {METRICS_CSV}")
    print(pd.read_csv(METRICS_CSV).to_string(index=False))


In [ ]:
# Графики
def plot_histories(results, title):
    wh = {k:v for k,v in results.items() if 'history' in v and v.get('status')=='ok'}
    if not wh: return
    n=len(wh); fig,axes=plt.subplots(n,3,figsize=(15,4*n))
    if n==1: axes=axes[np.newaxis,:]
    for ri,(rid,r) in enumerate(wh.items()):
        h=r['history']; ep=range(1,len(h['train_loss'])+1)
        axes[ri,0].plot(ep,h['train_loss']); axes[ri,0].set_title(f"{rid}: Train Loss"); axes[ri,0].grid(True,alpha=0.3)
        axes[ri,1].plot(ep,h['val_loss'],color='orange'); axes[ri,1].set_title(f"{rid}: Val Loss"); axes[ri,1].grid(True,alpha=0.3)
        axes[ri,2].plot(ep,h['val_cpc_full'],label='Full',color='green')
        axes[ri,2].plot(ep,h['val_cpc_nz'],label='NZ',color='blue',ls='--')
        axes[ri,2].set_title(f"{rid}: CPC"); axes[ri,2].set_ylim(0,1); axes[ri,2].legend(); axes[ri,2].grid(True,alpha=0.3)
    fig.suptitle(title,fontsize=14); plt.tight_layout(); plt.show()

plot_histories(single_city_results, "Single-city training")
plot_histories(multi_city_results, "Multi-city training")
